<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [1]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [ ]:
# Memilih dataset dan parameter pemisahan
import pandas as pd
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data, clean_data, split_data
from pci_ml.memory import configure, release
from pci_ml.ui import build_panel

RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

# Helper function: compact preview of the first rows
def format_adaptive_df(df_input):
    if df_input is None:
        return ""
    df_head = df_input.head()
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

df = None
handle = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_adaptive_df(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select)

handle = display(Markdown(""), display_id=True)

on_select(panel.file.value)


In [ ]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))

print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
      f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
      f"PCI minimum {cfg['pci_min']}")

markdown_table = format_adaptive_df(df)
display(Markdown(markdown_table))


#### 2. Pemisahan data latih dan data uji

In [ ]:
# Memisah data sesuai kriteria
if globals().get("df") is None:
    df, features = clean_data(load_data(filename))

def show_progress(done, total, best_gap):
    gap = "-" if best_gap is None else f"{best_gap:.4f}"
    print(f"\rMencari Seeds: {done}/{total} | Best Gap: {gap}", end="", flush=True)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=show_progress)
print()

split_summary = result["summary"]

# Print the summary outside the progress line so it always shows up
print(split_summary)

release("result", "df", "show_progress")


#### 3. Pelatihan model *machine learning*

In [ ]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import PCI, ROAD_ID, METADATA, latest_checkpoint, load_checkpoint

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

X_train = X_test = y_train = y_test = groups_train = test_ids = meta = None

def load_latest_ckpt(tag):
    global X_train, X_test, y_train, y_test, groups_train, test_ids, features, meta
    # train_df / test_df are local: they are freed when this function returns
    train_df, test_df, features, meta = load_checkpoint(filename, tag)

    X_train = train_df[features].to_numpy(dtype="float32")
    X_test  = test_df[features].to_numpy(dtype="float32")
    y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
    y_test  = test_df[PCI].to_numpy(dtype="float32")

    groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
    test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)   # labels for result tables

    assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

def split_info():
    n_tr, n_te = len(X_train), len(X_test)
    total = n_tr + n_te
    return (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
            f"**Test:** {X_test.shape} ({n_te / total:.2%})")

load_latest_ckpt(ckpt_tag)

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_info()}"))

release("df", "ckpt_tag", "load_latest_ckpt", "split_info", "current")


In [ ]:
# Verifikasi data input
import numpy as np
from pci_ml.data import ROAD_ID

# 1. Shapes and dtypes
print("Arrays:")
for name in ("X_train", "X_test", "y_train", "y_test", "groups_train"):
    a = globals()[name]
    print(f"  {name:<14}{str(a.shape):<12}{a.dtype}")
print(f"  {'test_ids':<14}{str(test_ids.shape):<12}columns: {list(test_ids.columns)}")
print(f"  {'features':<14}{len(features)} columns | first 3: {features[:3]}")

# 2. Consistency checks
checks = {
    "X columns == len(features)":        X_train.shape[1] == X_test.shape[1] == len(features),
    "X/y/groups rows match (train)":     len(X_train) == len(y_train) == len(groups_train),
    "X/y/test_ids rows match (test)":    len(X_test) == len(y_test) == len(test_ids),
    "row counts match meta":             (len(X_train), len(X_test)) == (meta["n_train"], meta["n_test"]),
    "no NaN/inf in arrays":              all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)),
    "X within 0-1":                      min(X_train.min(), X_test.min()) >= 0 and max(X_train.max(), X_test.max()) <= 1,
    "y within 0-100":                    min(y_train.min(), y_test.min()) >= 0 and max(y_train.max(), y_test.max()) <= 100,
    "no road shared by train and test":  set(groups_train).isdisjoint(test_ids[ROAD_ID]),
    "train_df/test_df released":         "train_df" not in globals() and "test_df" not in globals(),
}
print("\nChecks:")
for label, ok in checks.items():
    print(f"  [{'OK' if ok else 'FAIL'}] {label}")

# 3. Summary of the selected split
print(f"\nSeed split: {meta['seed']} | Gap: {meta['gap']:.4f} | Dibuat: {meta['created_at']}")
print(f"PCI train: mean {y_train.mean():.2f}, std {y_train.std(ddof=1):.2f} | "
      f"test: mean {y_test.mean():.2f}, std {y_test.std(ddof=1):.2f}")

### ***Source code* modul dan fungsi**

In [7]:
# import os
# from pathlib import Path
# from IPython.display import display, Markdown

# # Path to your src directory
# src_dir = Path("src")

# # Loop through all Python files and render them as Markdown code blocks
# for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
#     # Ignore any files inside Jupyter checkpoint folders
#     if ".ipynb_checkpoints" in filepath.parts:
#         continue

#     with open(filepath, "r", encoding="utf-8") as f:
#         code = f.read()
    
#     # Creates a nice header for the filename and formats the code
#     display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))